# Trendyol Datathon 2 — v6 Colab eğitimi (`bge-reranker-v2-m3`)

Hedef: 0.851 → 0.92+. Yöntem kararı ve gerekçesi repo'daki **HANDOFF.md bölüm 7**'de.

## Tek seferlik kurulum
1. Drive'da `MyDrive/trendyol_v6/` klasörü oluştur, içine koy:
   - `kaggle.json` (Kaggle → Settings → API → *Create New Token*)
   - `artifacts/train_dataset_v5.parquet` (laptoptaki `artifacts/` klasöründen, 22 MB)
   - isteğe bağlı: `artifacts/proxy_lists.parquet` (sanity raporu), `artifacts/ce_v3_test_mean.npy` (blend için)
2. Runtime → *Change runtime type* → **A100** (yoksa L4; T4 önerilmez).
3. Aşağıdaki *Parametreler* hücresinde `COMPETITION` slug'ını doldur (yarışma URL'sindeki ad).

## Her oturumda
Hücreleri baştan sona sırayla çalıştır. **Her adım resumable**: oturum koparsa yeniden bağlan ve hepsini tekrar çalıştır — eğitim son checkpoint'ten (≤20 dk kayıp), test skorlama son chunk'tan devam eder; biten adımlar kendini atlar.

## Süre tahmini (1.4M satır eğitim + 3.36M çift skorlama)
| GPU | eğitim | test skorlama |
|---|---|---|
| A100 40GB | ~3–4 saat | ~20–30 dk |
| L4 24GB | ~8–10 saat | ~1.5 saat |

## Submission planı (hepsi %25 oranla, her biri TEK değişken test eder)
1. `sub_v6_rate25.csv` — model sınıfı atlaması
2. `sub_v7_rate25.csv` — v6 öğretmenli pseudo-label döngüsü
3. blend (v6+v7, varsa +v3) — çeşitlilik

Kaggle'da **Select Submission** listesini güncel tutmayı unutma (final 2 seçim!).

In [ ]:
# 0) Parametreler
COMPETITION = ""   # Kaggle yarisma slug'i, or. "trendyol-datathon-2". Bos ise veri Drive'dan (WORK/data) kopyalanir.
REPO_URL = "https://github.com/emrehasilik/trendyol_kaggle.git"
GITHUB_TOKEN = ""  # repo private ise Personal Access Token, degilse bos
WORK = "/content/drive/MyDrive/trendyol_v6"
MODEL = "BAAI/bge-reranker-v2-m3"  # yedek aday: "microsoft/mdeberta-v3-base" (yalniz bf16 GPU: A100/L4)

In [ ]:
# 1) GPU kontrol
!nvidia-smi -L
import torch
bf16 = torch.cuda.get_device_capability(0)[0] >= 8
print(torch.cuda.get_device_name(0), "| bf16:", bf16, "| torch", torch.__version__)
if not bf16:
    print("UYARI: bf16 yok (T4?) -> fp16 yolu kullanilir; mumkunse A100/L4 sec.")

In [ ]:
# 2) Drive + kalici klasorler
from google.colab import drive
drive.mount("/content/drive")
import os
for d in ("artifacts", "models", "output"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)

In [ ]:
# 3) Repo + kutuphaneler
import os
url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
else:
    !cd /content/repo && git pull -q
%pip install -q "transformers>=4.44" sentencepiece pyarrow kaggle scikit-learn

In [ ]:
# 4) Ortam degiskenleri (src/config.py bunlari okur)
import os
os.environ["TK2_WORK"] = WORK                      # artifacts/models/output -> Drive (kalicilik)
os.environ["TK2_MODEL"] = MODEL
os.environ["TK2_HF_CACHE"] = "/content/hf_cache"   # HF model cache Drive kotasini yemesin
print("WORK =", WORK, "| MODEL =", MODEL)

In [ ]:
# 5) Yarisma verisi (Kaggle API; slug bos ise Drive fallback)
import os
DATA = "/content/repo/data"
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f"{DATA}/items.csv"):
    if COMPETITION:
        os.makedirs("/root/.kaggle", exist_ok=True)
        !cp {WORK}/kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
        !kaggle competitions download -c {COMPETITION} -p {DATA}
        !cd {DATA} && unzip -o -q "*.zip" && rm -f *.zip
    else:
        # fallback: 4 CSV'yi Drive'da WORK/data altina onceden yukle
        !cp {WORK}/data/*.csv {DATA}/
!ls -lh {DATA}

In [ ]:
# 6) Egitim verisi kontrolu
import os
p = f"{WORK}/artifacts/train_dataset_v5.parquet"
assert os.path.exists(p), f"EKSIK: {p} — laptoptaki artifacts/train_dataset_v5.parquet dosyasini Drive'a yukle"
print("OK:", p)
for f in ("proxy_lists.parquet", "ce_v3_test_mean.npy"):
    ok = os.path.exists(f"{WORK}/artifacts/{f}")
    print(("OK: " if ok else "yok (istege bagli): ") + f)

In [ ]:
# 7) Token cache (tokenizer'a ozel; varsa kendini atlar, ~5-10 dk)
!cd /content/repo/src && python -u tokenize_ce_cache_v6.py

## Eğitim
İlk 50 adımda hız/VRAM/loss ölçümü basılır (`[smoke]` satırı) — **ETA'yı ve loss'un düştüğünü orada doğrula**, sorun varsa erken durdur. Oturum koparsa bu hücreyi yeniden çalıştırman yeterli; son checkpoint'ten sürer.

In [ ]:
# 8) v6 egitimi (resumable)
!cd /content/repo/src && python -u train_ce_v6.py --data train_dataset_v5.parquet --tag v6

In [ ]:
# 9) Test skorlama + %25 oranli submission (chunk-bazli resumable)
!cd /content/repo/src && python -u infer_ce_v6.py --tag v6 --rate 0.25

In [ ]:
# 10) Submission gonder
print("dosya:", f"{WORK}/output/sub_v6_rate25.csv")
# API ile gondermek icin asagiyi ac (COMPETITION dolu olmali):
# !kaggle competitions submit -c {COMPETITION} -f {WORK}/output/sub_v6_rate25.csv -m "v6 bge-reranker-v2-m3 rate25"
# Sonra Kaggle'da 'Select Submission' listesini guncelle (final 2 secim!)

## v7 — pseudo-label döngüsü (yeni öğretmen)
**v6'nın LB sonucu sıçramayı doğruladıktan sonra** çalıştır. v6 farklı model sınıfı olduğundan öğretmen değişti → döngü gerçekten yeni bilgi taşır (HANDOFF DERS #3/#5'e takılmaz).

In [ ]:
# 11) v7 dataseti (capalar + v6 ogretmenli pseudo etiketler)
!cd /content/repo/src && python -u build_dataset_v7.py

In [ ]:
# 12) v7 egitimi (ayni script, farkli veri/tag; resumable)
!cd /content/repo/src && python -u train_ce_v6.py --data train_dataset_v7.parquet --tag v7

In [ ]:
# 13) v7 test skorlama + submission
!cd /content/repo/src && python -u infer_ce_v6.py --tag v7 --rate 0.25

In [ ]:
# 14) Blend: v6+v7 (varsa +v3 eski-aile cesitliligi), %25 oranla
import os
import numpy as np, pandas as pd
A = f"{WORK}/artifacts"
s6 = np.load(f"{A}/ce_v6_test.npy"); s7 = np.load(f"{A}/ce_v7_test.npy")
blend, name = (s6 + s7) / 2, "v67"
if os.path.exists(f"{A}/ce_v3_test_mean.npy"):
    blend = 0.4 * s6 + 0.4 * s7 + 0.2 * np.load(f"{A}/ce_v3_test_mean.npy")
    name = "v673"
thr = float(np.quantile(blend, 0.75))
ids = pd.read_csv("/content/repo/data/submission_pairs.csv", usecols=["id"], dtype=str)["id"]
out = f"{WORK}/output/sub_{name}blend_rate25.csv"
pd.DataFrame({"id": ids, "prediction": (blend > thr).astype(int)}).to_csv(out, index=False)
print("yazildi:", out, "| esik:", round(thr, 4))